# PCA Dimensionality Reduction — HAR Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
import joblib

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

TRAIN_PATH = '/mnt/user-data/uploads/train.csv'
TEST_PATH  = '/mnt/user-data/uploads/test.csv'
TARGET_COL = 'Activity'
DROP_COLS  = ['subject']
RANDOM_STATE = 42

## 1. Data Loading & Inspection

In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

df = pd.concat([train_raw, test_raw], ignore_index=True)

print("Combined shape:", df.shape)
print("\nColumn sample (first 5):", list(df.columns[:5]))
print("Column sample (last 3) :", list(df.columns[-3:]))
print("\nData types summary:")
print(df.dtypes.value_counts())

## 2. Data Exploration

In [ ]:
feature_cols = [c for c in df.columns if c not in [TARGET_COL] + DROP_COLS]
print(f"Total features  : {len(feature_cols)}")
print(f"Target column   : {TARGET_COL}")
print(f"Total rows      : {len(df)}")

missing = df[feature_cols].isnull().sum()
print(f"\nMissing values per column (non-zero only):")
print(missing[missing > 0] if missing.any() else "  None")

print("\nClass distribution:")
print(df[TARGET_COL].value_counts())

## 3. Preprocessing

In [ ]:
thresh = df.shape[1] * 0.5
df = df.dropna(thresh=int(thresh))

num_cols = [c for c in feature_cols if df[c].dtype in ['float64','float32','int64','int32']]
cat_cols = [c for c in feature_cols if c not in num_cols]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    feature_cols = [c for c in df.columns if c not in [TARGET_COL] + DROP_COLS]
    num_cols = feature_cols

le = LabelEncoder()
y = le.fit_transform(df[TARGET_COL])
class_names = le.classes_

X = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Preprocessed feature matrix shape:", X_scaled.shape)
print("Classes:", class_names)

## 4. Exploratory Plots

In [ ]:
sample_feats = feature_cols[:12]
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, col in zip(axes.flatten(), sample_feats):
    ax.hist(df[col].values, bins=40, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(col[:30], fontsize=7)
    ax.set_yticks([])
plt.suptitle("Feature Distributions (first 12 features)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
corr_sample = pd.DataFrame(X_scaled[:, :20], columns=feature_cols[:20])
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_sample.corr(), dtype=bool))
sns.heatmap(corr_sample.corr(), mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title("Correlation Heatmap (first 20 features)", fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
counts = df[TARGET_COL].value_counts()
ax.barh(counts.index, counts.values, color='steelblue')
ax.set_xlabel("Count")
ax.set_title("Class Distribution")
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. PCA Pipeline

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

evr = pca_full.explained_variance_ratio_
cum_evr = np.cumsum(evr)

n_components_95 = int(np.argmax(cum_evr >= 0.95)) + 1
print(f"Components needed for 95% variance: {n_components_95}")

checkpoints = [c for c in [5, 10, 20, 50, 100] if c <= X_scaled.shape[1]]
print("\nComponents -> Cumulative Variance:")
for n in checkpoints:
    print(f"  Components {n:>3} -> {cum_evr[n-1]*100:.1f}%")
print(f"  Components {n_components_95:>3} -> {cum_evr[n_components_95-1]*100:.1f}%  (95% threshold)")

In [ ]:
variance_table = pd.DataFrame({
    'n_components': checkpoints + [n_components_95],
    'cumulative_variance_pct': [round(cum_evr[n-1]*100, 2) for n in checkpoints] + [round(cum_evr[n_components_95-1]*100, 2)]
}).drop_duplicates('n_components').sort_values('n_components').reset_index(drop=True)
print(variance_table.to_string(index=False))

## 6. Explained Variance Plot

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(range(1, len(cum_evr)+1), cum_evr * 100, linewidth=1.5, color='steelblue')
ax.axhline(95, color='red', linestyle='--', linewidth=1, label='95% threshold')
ax.axvline(n_components_95, color='orange', linestyle='--', linewidth=1,
           label=f'{n_components_95} components')
ax.scatter([n_components_95], [cum_evr[n_components_95-1]*100], color='orange', zorder=5, s=60)
ax.set_xlabel("Number of Components")
ax.set_ylabel("Cumulative Explained Variance (%)")
ax.set_title("PCA Cumulative Explained Variance")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pca_variance_curve.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Dimensionality Reduction Outputs

In [ ]:
pca_95 = PCA(n_components=n_components_95, random_state=RANDOM_STATE)
X_pca_95 = pca_95.fit_transform(X_scaled)

pca_reduced_df = pd.DataFrame(X_pca_95, columns=[f'PC{i+1}' for i in range(n_components_95)])
pca_reduced_df[TARGET_COL] = df[TARGET_COL].values
pca_reduced_df.to_csv('pca_reduced.csv', index=False)
print("Saved pca_reduced.csv  shape:", pca_reduced_df.shape)

for n in [2, 5, 10, 20]:
    if n <= X_scaled.shape[1]:
        pca_n = PCA(n_components=n, random_state=RANDOM_STATE)
        X_n = pca_n.fit_transform(X_scaled)
        out = pd.DataFrame(X_n, columns=[f'PC{i+1}' for i in range(n)])
        out.to_csv(f'pca_{n}.csv', index=False)
        print(f"Saved pca_{n}.csv  shape: {out.shape}")

## 8. 2D & 3D Visualization

In [ ]:
pca_2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2 = pca_2.fit_transform(X_scaled)

palette = sns.color_palette('tab10', len(class_names))
fig, ax = plt.subplots(figsize=(10, 7))
for i, cls in enumerate(class_names):
    mask = y == i
    ax.scatter(X_pca_2[mask, 0], X_pca_2[mask, 1],
               label=cls, alpha=0.5, s=8, color=palette[i])
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({evr[1]*100:.1f}%)")
ax.set_title("PCA 2D — Colored by Activity Class")
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig('pca_2d_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

pca_3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca_3 = pca_3.fit_transform(X_scaled)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
for i, cls in enumerate(class_names):
    mask = y == i
    ax.scatter(X_pca_3[mask, 0], X_pca_3[mask, 1], X_pca_3[mask, 2],
               label=cls, alpha=0.4, s=5, color=palette[i])
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
ax.set_title("PCA 3D")
ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.savefig('pca_3d_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. t-SNE

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, n_jobs=-1)
X_tsne = tsne.fit_transform(X_scaled)
print("t-SNE output shape:", X_tsne.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for i, cls in enumerate(class_names):
    mask = y == i
    axes[0].scatter(X_pca_2[mask, 0], X_pca_2[mask, 1],
                    label=cls, alpha=0.5, s=8, color=palette[i])
axes[0].set_title("PCA 2D")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(markerscale=3, fontsize=8)

for i, cls in enumerate(class_names):
    mask = y == i
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    label=cls, alpha=0.5, s=8, color=palette[i])
axes[1].set_title("t-SNE 2D")
axes[1].set_xlabel("Dim 1"); axes[1].set_ylabel("Dim 2")
axes[1].legend(markerscale=3, fontsize=8)

plt.suptitle("PCA vs t-SNE — Side by Side", fontsize=14)
plt.tight_layout()
plt.savefig('pca_vs_tsne.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Before vs After ML Comparison

In [ ]:
X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

X_train_pca = X_pca_95[:len(X_train_orig) + len(X_test_orig)]
X_train_pca_split, X_test_pca_split, _, _ = train_test_split(
    X_pca_95, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

def get_memory_mb():
    if HAS_PSUTIL:
        return psutil.Process().memory_info().rss / 1e6
    return None

clf_orig = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
mem_before = get_memory_mb()
t0 = time.time()
clf_orig.fit(X_train_orig, y_train)
time_orig = time.time() - t0
mem_after_orig = get_memory_mb()
acc_orig = clf_orig.score(X_test_orig, y_test)

clf_pca = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
t1 = time.time()
clf_pca.fit(X_train_pca_split, y_train)
time_pca = time.time() - t1
mem_after_pca = get_memory_mb()
acc_pca = clf_pca.score(X_test_pca_split, y_test)

mem_orig_str = f"{mem_after_orig - mem_before:.1f} MB" if HAS_PSUTIL else "N/A"
mem_pca_str  = f"{mem_after_pca - mem_after_orig:.1f} MB" if HAS_PSUTIL else "N/A"

comparison_df = pd.DataFrame({
    'Metric':    ['Features', 'Training Time (s)', 'Test Accuracy', 'Memory Usage'],
    'Original':  [X_train_orig.shape[1], f'{time_orig:.2f}', f'{acc_orig:.4f}', mem_orig_str],
    'PCA-Reduced': [n_components_95, f'{time_pca:.2f}', f'{acc_pca:.4f}', mem_pca_str]
})
print(comparison_df.to_string(index=False))

## 11. Feature Reduction Analysis

In [ ]:
original_features = X_scaled.shape[1]
reduced_features  = n_components_95
variance_preserved = round(cum_evr[n_components_95-1] * 100, 2)
time_reduction_pct = round((1 - time_pca / time_orig) * 100, 1) if time_orig > 0 else 0.0

print(f"Original Features  : {original_features}")
print(f"Reduced Features   : {reduced_features}")
print(f"Variance Preserved : {variance_preserved}%")
print(f"Training Time Orig : {time_orig:.2f}s  |  PCA: {time_pca:.2f}s")
print(f"Training Time Reduced: {time_reduction_pct}%")

## 12. PCA vs t-SNE Comparison Table

In [ ]:
pca_tsne_df = pd.DataFrame({
    'Aspect'      : ['Purpose', 'Linearity', 'Speed', 'Reproducibility', 'Best Use Case'],
    'PCA'         : ['Variance maximisation', 'Linear', 'Fast (O(n*d^2))', 'Deterministic',
                     'Preprocessing, noise reduction, speed-up ML'],
    't-SNE'       : ['Cluster visualisation', 'Non-linear', 'Slow (O(n^2))', 'Stochastic (seed helps)',
                     '2D/3D exploratory visualisation of clusters']
})
print(pca_tsne_df.to_string(index=False))

## 13. Interactive Component Selection

In [ ]:
component_results = []
for n in [2, 5, 10, 20]:
    if n <= X_scaled.shape[1]:
        pca_n = PCA(n_components=n, random_state=RANDOM_STATE)
        X_n = pca_n.fit_transform(X_scaled)
        X_tr, X_te, y_tr, y_te = train_test_split(X_n, y, test_size=0.2,
                                                    stratify=y, random_state=RANDOM_STATE)
        clf_n = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
        t_s = time.time()
        clf_n.fit(X_tr, y_tr)
        t_e = time.time()
        acc_n = clf_n.score(X_te, y_te)
        component_results.append({
            'n_components': n,
            'variance_pct': round(np.sum(pca_n.explained_variance_ratio_)*100, 2),
            'train_time_s': round(t_e - t_s, 3),
            'test_accuracy': round(acc_n, 4)
        })

comp_df = pd.DataFrame(component_results)
print(comp_df.to_string(index=False))

## 14. Cluster Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for i, cls in enumerate(class_names):
    mask = y == i
    axes[0].scatter(X_pca_2[mask, 0], X_pca_2[mask, 1],
                    label=cls, alpha=0.5, s=8, color=palette[i])
axes[0].set_title("PCA 2D — True Activity Labels")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(markerscale=3, fontsize=8)

km = KMeans(n_clusters=6, random_state=RANDOM_STATE, n_init='auto')
km_labels = km.fit_predict(X_pca_2)
sil_score = silhouette_score(X_pca_2, km_labels, sample_size=3000, random_state=RANDOM_STATE)
km_palette = sns.color_palette('Set2', 6)
for k in range(6):
    mask = km_labels == k
    axes[1].scatter(X_pca_2[mask, 0], X_pca_2[mask, 1],
                    label=f'Cluster {k}', alpha=0.5, s=8, color=km_palette[k])
axes[1].set_title(f"PCA 2D — KMeans (k=6)  Silhouette={sil_score:.3f}")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].legend(markerscale=3, fontsize=8)

plt.suptitle("Cluster Visualization", fontsize=14)
plt.tight_layout()
plt.savefig('cluster_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Silhouette Score (KMeans k=6): {sil_score:.4f}")

## 15. Final PCA Report

In [ ]:
pca_report = {
    'original_features'          : original_features,
    'optimal_components_for_95pct': n_components_95,
    'variance_preserved_pct'      : variance_preserved,
    'training_time_reduction_pct' : time_reduction_pct,
    'accuracy_before'             : round(acc_orig, 4),
    'accuracy_after'              : round(acc_pca, 4),
}
for k, v in pca_report.items():
    print(f"  {k}: {v}")

## 16. Save Models

In [ ]:
joblib.dump(pca_95,    'pca_model.pkl')
joblib.dump(clf_orig,  'classifier_model.pkl')
print("Saved: pca_model.pkl")
print("Saved: classifier_model.pkl")
print("\nAll output files:")
import os
for f in sorted(os.listdir('.')):
    if f.endswith(('.csv','.pkl','.png')):
        print(f"  {f}  ({os.path.getsize(f)/1024:.1f} KB)")